# 🚢 Titanic Survival Prediction

## CodSoft Data Science Internship — Task 1

---

### 📋 Introduction

The sinking of the RMS Titanic on April 15, 1912, is one of the most infamous shipwrecks in history. Of the 2,224 passengers and crew aboard, more than 1,500 died. This project uses Machine Learning to predict which passengers survived the tragedy based on features like passenger class, gender, age, fare, and family details.

### 🎯 Business Problem

**Goal:** Build a classification model that predicts whether a given passenger survived the Titanic disaster (binary classification: 0 = Not Survived, 1 = Survived).

**Why it matters:** Understanding survival patterns helps analyze emergency response, socio-economic factors in disasters, and builds foundational ML skills.

### 📊 Dataset

- **Source:** [Kaggle — Titanic Dataset (yasserh)](https://www.kaggle.com/datasets/yasserh/titanic-dataset)
- **Size:** 891 passengers × 12 features
- **Target:** `Survived` (0 or 1)

---

## 1. Import Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import warnings
import os
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Model persistence
import joblib

# Settings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

%matplotlib inline

print('All libraries imported successfully!')

---

## 2. Setup Directories

In [ ]:
# Project paths
PROJECT_ROOT = Path('..').resolve()
DATASET_DIR = PROJECT_ROOT / 'dataset'
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'

# Create directories
for d in [MODELS_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Project Root: {PROJECT_ROOT}')
print(f'Dataset Dir:  {DATASET_DIR}')
print(f'Models Dir:   {MODELS_DIR}')
print(f'Outputs Dir:  {OUTPUTS_DIR}')

---

## 3. Load Dataset

We load the Titanic dataset downloaded from [Kaggle (yasserh)](https://www.kaggle.com/datasets/yasserh/titanic-dataset).

In [ ]:
df = pd.read_csv(DATASET_DIR / 'Titanic-Dataset.csv')
print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(10)

---

## 4. Dataset Exploration

Let's thoroughly inspect the dataset to understand its structure, types, and quality.

### 4.1 Shape & Columns

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')

### 4.2 Data Types

In [ ]:
df.dtypes

In [ ]:
df.info()

### 4.3 Summary Statistics

Statistical summary of numerical columns gives us insight into the distribution of values.

In [ ]:
df.describe().round(2)

In [ ]:
# Categorical columns summary
df.describe(include='object')

### 4.4 Missing Values

Understanding missing data is critical before any preprocessing.

In [ ]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
missing[missing['Missing Count'] > 0].sort_values('Missing %', ascending=False)

**Observations:**
- **Cabin** has ~77% missing values — too many to impute with original values. We'll create a binary flag.
- **Age** has ~20% missing — we'll impute using median grouped by Pclass and Sex.
- **Embarked** has only 2 missing — mode imputation is appropriate.

### 4.5 Duplicate Records

In [ ]:
print(f'Duplicate rows: {df.duplicated().sum()}')

### 4.6 Target Distribution

In [ ]:
print('Survival Distribution:')
print(df['Survived'].value_counts())
print(f'\nSurvival Rate: {df["Survived"].mean():.2%}')

**Observation:** ~38.4% survived. The dataset has moderate class imbalance but not severe enough to require special handling like SMOTE.

### 4.7 Unique Values per Column

In [ ]:
for col in df.columns:
    print(f'{col:15s}: {df[col].nunique():4d} unique values')

### 4.8 Correlation Matrix

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
numeric_df.corr().round(2)

---

## 5. Exploratory Data Analysis (EDA)

Visualizations to understand patterns and relationships in the data.

### 5.1 Missing Values Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=ax)
ax.set_title('Missing Values Heatmap', fontsize=15, fontweight='bold')
fig.savefig(OUTPUTS_DIR / 'missing_values_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation Heatmap', fontsize=15, fontweight='bold')
fig.savefig(OUTPUTS_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

**Key correlations with Survived:**
- **Pclass** (negative): Higher class number = lower class = lower survival
- **Fare** (positive): Higher fare = likely higher class = better survival
- **Parch/SibSp** (slight positive): Some family presence helped

### 5.3 Survival Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(x='Survived', data=df, palette=['#ef4444', '#22c55e'], ax=ax)
ax.set_title('Survival Distribution', fontsize=15, fontweight='bold')
ax.set_xticklabels(['Not Survived (0)', 'Survived (1)'])
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
               ha='center', va='bottom', fontsize=13)
fig.savefig(OUTPUTS_DIR / 'survival_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Passenger Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(x='Pclass', data=df, palette='Set2', ax=ax)
ax.set_title('Passenger Class Distribution', fontsize=15, fontweight='bold')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
               ha='center', va='bottom', fontsize=13)
fig.savefig(OUTPUTS_DIR / 'pclass_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 Gender Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(x='Sex', data=df, palette=['#3b82f6', '#ec4899'], ax=ax)
ax.set_title('Gender Distribution', fontsize=15, fontweight='bold')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
               ha='center', va='bottom', fontsize=13)
fig.savefig(OUTPUTS_DIR / 'gender_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.6 Age Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df['Age'].dropna(), bins=30, kde=True, color='#8b5cf6', ax=ax)
ax.set_title('Age Distribution', fontsize=15, fontweight='bold')
ax.axvline(df['Age'].median(), color='red', linestyle='--', label=f'Median: {df["Age"].median():.1f}')
ax.legend()
fig.savefig(OUTPUTS_DIR / 'age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.7 Fare Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df['Fare'].dropna(), bins=30, kde=True, color='#f59e0b', ax=ax)
ax.set_title('Fare Distribution', fontsize=15, fontweight='bold')
ax.axvline(df['Fare'].median(), color='red', linestyle='--', label=f'Median: {df["Fare"].median():.1f}')
ax.legend()
fig.savefig(OUTPUTS_DIR / 'fare_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.8 Survival vs Gender

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(x='Sex', hue='Survived', data=df, palette=['#ef4444', '#22c55e'], ax=ax)
ax.set_title('Survival vs Gender', fontsize=15, fontweight='bold')
ax.legend(['Not Survived', 'Survived'])
fig.savefig(OUTPUTS_DIR / 'survival_vs_gender.png', dpi=150, bbox_inches='tight')
plt.show()

**Observation:** Females had significantly higher survival rates than males. This is consistent with the "women and children first" evacuation policy.

### 5.9 Survival vs Passenger Class

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(x='Pclass', hue='Survived', data=df, palette=['#ef4444', '#22c55e'], ax=ax)
ax.set_title('Survival vs Passenger Class', fontsize=15, fontweight='bold')
ax.legend(['Not Survived', 'Survived'])
fig.savefig(OUTPUTS_DIR / 'survival_vs_pclass.png', dpi=150, bbox_inches='tight')
plt.show()

**Observation:** 1st class passengers had much higher survival rates. 3rd class had the lowest survival — likely due to cabin location (lower decks, farther from lifeboats).

### 5.10 Age vs Survival

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
survived = df[df['Survived'] == 1]['Age'].dropna()
not_survived = df[df['Survived'] == 0]['Age'].dropna()
ax.hist(not_survived, bins=30, alpha=0.6, color='#ef4444', label='Not Survived', edgecolor='white')
ax.hist(survived, bins=30, alpha=0.6, color='#22c55e', label='Survived', edgecolor='white')
ax.set_xlabel('Age', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
ax.set_title('Age vs Survival', fontsize=15, fontweight='bold')
ax.legend()
fig.savefig(OUTPUTS_DIR / 'age_vs_survival.png', dpi=150, bbox_inches='tight')
plt.show()

**Observation:** Young children (under ~10) had notably higher survival rates. Elderly passengers had lower survival. The 20-30 age group had the most casualties.

---

## 6. Data Cleaning

Now we handle missing values, duplicates, and prepare the data for modeling.

In [ ]:
# Create a working copy
df_clean = df.copy()
print(f'Starting shape: {df_clean.shape}')
print(f'Missing values before cleaning:\n{df_clean.isnull().sum()[df_clean.isnull().sum() > 0]}')

### 6.1 Handle Missing Age

**Strategy:** Fill with median Age grouped by Pclass and Sex. This is more accurate than a global median because age distribution varies significantly by class and gender.

In [ ]:
# Show median ages by group
print('Median Age by Pclass & Sex:')
print(df_clean.groupby(['Pclass', 'Sex'])['Age'].median())

# Impute
df_clean['Age'] = df_clean.groupby(['Pclass', 'Sex'])['Age'].transform(
    lambda x: x.fillna(x.median())
)
# Fallback for edge cases
df_clean['Age'].fillna(df_clean['Age'].median(), inplace=True)

print(f'\nAge missing after imputation: {df_clean["Age"].isnull().sum()}')

### 6.2 Handle Missing Embarked

**Strategy:** Only 2 values missing. Fill with the mode ('S' = Southampton, the most common port).

In [ ]:
print(f'Embarked value counts:\n{df_clean["Embarked"].value_counts()}')
df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0], inplace=True)
print(f'\nEmbarked missing after imputation: {df_clean["Embarked"].isnull().sum()}')

### 6.3 Handle Missing Cabin

**Strategy:** With ~77% missing, imputing actual cabin values is unreliable. Instead, we mark it as 'Unknown' and later create a binary `CabinAvailable` feature.

In [ ]:
df_clean['Cabin'].fillna('Unknown', inplace=True)
print(f'Cabin missing after filling: {df_clean["Cabin"].isnull().sum()}')

### 6.4 Remove Duplicates

In [ ]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f'Rows before: {before}, Rows after: {len(df_clean)}, Removed: {before - len(df_clean)}')

### 6.5 Verify Clean Data

In [ ]:
print('Missing values after cleaning:')
print(df_clean.isnull().sum())
print(f'\nFinal shape: {df_clean.shape}')

---

## 7. Feature Engineering

Creating new features from existing data to improve model performance.

### 7.1 Title (from Name)

**Why:** The title encodes social status, gender, and approximate age — all strong predictors of survival. For example, 'Mrs' implies a married woman (higher survival), while 'Mr' implies an adult male (lower survival).

In [ ]:
# Extract title from Name
df_clean['Title'] = df_clean['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
print('Raw title distribution:')
print(df_clean['Title'].value_counts())

In [ ]:
# Consolidate rare titles
title_mapping = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Countess': 'Rare', 'Ms': 'Miss', 'Lady': 'Rare',
    'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare', 'Mme': 'Mrs',
    'Capt': 'Rare', 'Sir': 'Rare'
}
df_clean['Title'] = df_clean['Title'].map(title_mapping).fillna('Rare')
print('\nConsolidated title distribution:')
print(df_clean['Title'].value_counts())

### 7.2 FamilySize

**Why:** Family size significantly affected survival. Small families (2-4) had better survival than solo travelers or very large families (coordination difficulties).

In [ ]:
df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1
print('FamilySize distribution:')
print(df_clean['FamilySize'].value_counts().sort_index())

### 7.3 IsAlone

**Why:** Solo travelers had lower survival rates. This binary feature captures the key signal from FamilySize.

In [ ]:
df_clean['IsAlone'] = (df_clean['FamilySize'] == 1).astype(int)
print(f'IsAlone distribution:\n{df_clean["IsAlone"].value_counts()}')
print(f'\nSurvival rate - Alone: {df_clean[df_clean["IsAlone"]==1]["Survived"].mean():.2%}')
print(f'Survival rate - With family: {df_clean[df_clean["IsAlone"]==0]["Survived"].mean():.2%}')

### 7.4 CabinAvailable

**Why:** Having a cabin number recorded may indicate higher-class passengers whose information was better documented.

In [ ]:
df_clean['CabinAvailable'] = (df_clean['Cabin'] != 'Unknown').astype(int)
print(f'CabinAvailable distribution:\n{df_clean["CabinAvailable"].value_counts()}')
print(f'\nSurvival rate - Cabin known: {df_clean[df_clean["CabinAvailable"]==1]["Survived"].mean():.2%}')
print(f'Survival rate - Cabin unknown: {df_clean[df_clean["CabinAvailable"]==0]["Survived"].mean():.2%}')

### 7.5 TicketFrequency

**Why:** Multiple passengers sharing a ticket number indicates families/groups traveling together, which affected survival dynamics.

In [ ]:
ticket_counts = df_clean['Ticket'].value_counts()
df_clean['TicketFrequency'] = df_clean['Ticket'].map(ticket_counts)
print(f'TicketFrequency distribution:\n{df_clean["TicketFrequency"].value_counts().sort_index()}')

### 7.6 FareCategory

**Why:** Fare is a proxy for socio-economic status. Binning reduces noise from extreme outliers while preserving the survival signal.

In [ ]:
df_clean['FareCategory'] = pd.qcut(df_clean['Fare'], q=4, labels=['Low', 'Medium', 'High', 'VeryHigh']).astype(str)
print(f'FareCategory distribution:\n{df_clean["FareCategory"].value_counts()}')

### 7.7 AgeCategory

**Why:** Children had priority for lifeboats ('women and children first'). Age groups capture this non-linear survival pattern.

In [ ]:
bins = [0, 12, 18, 60, 100]
labels = ['Child', 'Teen', 'Adult', 'Senior']
df_clean['AgeCategory'] = pd.cut(df_clean['Age'], bins=bins, labels=labels).astype(str)
print(f'AgeCategory distribution:\n{df_clean["AgeCategory"].value_counts()}')

### 7.8 Encode Engineered Categorical Features

In [ ]:
# One-hot encode FareCategory and AgeCategory
for col in ['FareCategory', 'AgeCategory']:
    dummies = pd.get_dummies(df_clean[col], prefix=col, dtype=int)
    df_clean = pd.concat([df_clean, dummies], axis=1)
    df_clean.drop(col, axis=1, inplace=True)

print(f'Shape after feature engineering: {df_clean.shape}')
df_clean.head()

---

## 8. Encoding & Final Preprocessing

Encode remaining categorical variables and drop columns not useful for modeling.

### 8.1 Encode Sex

**Label Encoding** is appropriate because Sex is binary (male/female).

In [ ]:
le_sex = LabelEncoder()
df_clean['Sex'] = le_sex.fit_transform(df_clean['Sex'])
print(f'Sex encoding: {dict(zip(le_sex.classes_, le_sex.transform(le_sex.classes_)))}')

### 8.2 Encode Embarked

**One-Hot Encoding** is appropriate because Embarked has 3 categories with no ordinal relationship.

In [ ]:
embarked_dummies = pd.get_dummies(df_clean['Embarked'], prefix='Embarked', dtype=int)
df_clean = pd.concat([df_clean, embarked_dummies], axis=1)
df_clean.drop('Embarked', axis=1, inplace=True)
print(f'Embarked encoded into: {list(embarked_dummies.columns)}')

### 8.3 Encode Title

In [ ]:
le_title = LabelEncoder()
df_clean['Title'] = le_title.fit_transform(df_clean['Title'])
print(f'Title encoding: {dict(zip(le_title.classes_, le_title.transform(le_title.classes_)))}')

### 8.4 Drop Unnecessary Columns

These columns are either identifiers, unstructured text, or have been replaced by engineered features.

In [ ]:
cols_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df_clean = df_clean.drop(columns=cols_to_drop)
print(f'Dropped: {cols_to_drop}')
print(f'\nFinal shape: {df_clean.shape}')
print(f'\nFinal columns ({len(df_clean.columns)}):')
for i, col in enumerate(df_clean.columns, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
df_clean.head()

---

## 9. Train-Test Split & Scaling

In [ ]:
# Split
X = df_clean.drop(columns=['Survived'])
y = df_clean['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape}')
print(f'Test set:     {X_test.shape}')
print(f'\nTarget distribution (train):\n{y_train.value_counts()}')
print(f'\nTarget distribution (test):\n{y_test.value_counts()}')

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print('Features scaled with StandardScaler.')
X_train_scaled.head()

---

## 10. Model Training & Comparison

We train 6 different classification algorithms and compare their performance.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
}

results = []

print(f'{"Model":25s} | {"Train Acc":>10s} | {"Test Acc":>10s} | {"CV Mean":>10s} | {"CV Std":>10s}')
print('-' * 80)

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    train_acc = model.score(X_train_scaled, y_train)
    test_acc = accuracy_score(y_test, y_pred)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    
    results.append({
        'Model': name,
        'Train Accuracy': round(train_acc, 4),
        'Test Accuracy': round(test_acc, 4),
        'CV Mean': round(cv_scores.mean(), 4),
        'CV Std': round(cv_scores.std(), 4),
    })
    
    print(f'{name:25s} | {train_acc:>10.4f} | {test_acc:>10.4f} | {cv_scores.mean():>10.4f} | {cv_scores.std():>10.4f}')

comparison_df = pd.DataFrame(results).sort_values('CV Mean', ascending=False).reset_index(drop=True)
comparison_df.to_csv(OUTPUTS_DIR / 'model_comparison.csv', index=False)
print('\n--- Model Comparison Table ---')
comparison_df

---

## 11. Hyperparameter Tuning

We perform GridSearchCV on the top 2 performing models to find optimal hyperparameters.

In [ ]:
top_models = comparison_df.head(2)['Model'].tolist()
print(f'Top models for tuning: {top_models}')

In [ ]:
param_grids = {
    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs'],
    },
    'Decision Tree': {
        'max_depth': [3, 5, 7, 10, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
    },
    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [5, 10, 15, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2],
    },
    'Gradient Boosting': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7],
        'min_samples_split': [2, 5],
    },
    'KNN': {
        'n_neighbors': [3, 5, 7, 9, 11],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan'],
    },
    'SVM': {
        'C': [0.1, 1, 10],
        'kernel': ['rbf', 'linear'],
        'gamma': ['scale', 'auto'],
    },
}

best_model = None
best_score = 0
best_name = ''
best_params = {}

all_base_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(probability=True, random_state=42),
}

for model_name in top_models:
    if model_name not in param_grids:
        continue
    print(f'\nTuning: {model_name}...')
    grid = GridSearchCV(
        all_base_models[model_name],
        param_grids[model_name],
        cv=5, scoring='accuracy', n_jobs=-1, verbose=0
    )
    grid.fit(X_train_scaled, y_train)
    print(f'  Best Parameters: {grid.best_params_}')
    print(f'  Best CV Score:   {grid.best_score_:.4f}')
    
    if grid.best_score_ > best_score:
        best_score = grid.best_score_
        best_model = grid.best_estimator_
        best_name = model_name
        best_params = grid.best_params_

print(f'\n{"="*60}')
print(f'BEST MODEL: {best_name}')
print(f'BEST CV SCORE: {best_score:.4f}')
print(f'BEST PARAMETERS: {best_params}')
print(f'{"="*60}')

---

## 12. Model Evaluation

Comprehensive evaluation of the best model with all relevant metrics.

In [ ]:
# Generate predictions
y_pred = best_model.predict(X_test_scaled)
y_proba = best_model.predict_proba(X_test_scaled)[:, 1] if hasattr(best_model, 'predict_proba') else None

# Calculate metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else None
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=5, scoring='accuracy')

print(f'Model: {best_name}')
print(f'{"─"*50}')
print(f'Accuracy:   {acc:.4f}  → Proportion of correct predictions overall')
print(f'Precision:  {prec:.4f}  → Of predicted survived, how many actually survived')
print(f'Recall:     {rec:.4f}  → Of actual survivors, how many were correctly identified')
print(f'F1 Score:   {f1:.4f}  → Harmonic mean of precision and recall')
if roc_auc:
    print(f'ROC-AUC:    {roc_auc:.4f}  → Ability to distinguish between classes')
print(f'CV Score:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}  → Generalization measure')

In [ ]:
# Classification Report
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Survived', 'Survived']))

### 12.1 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Survived', 'Survived'],
            yticklabels=['Not Survived', 'Survived'],
            ax=ax, annot_kws={'size': 16})
ax.set_xlabel('Predicted', fontsize=13)
ax.set_ylabel('Actual', fontsize=13)
ax.set_title('Confusion Matrix', fontsize=15, fontweight='bold')
fig.savefig(OUTPUTS_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 12.2 ROC Curve

In [ ]:
if y_proba is not None:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, color='#2563eb', lw=2, label=f'ROC Curve (AUC = {auc:.4f})')
    ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random Classifier')
    ax.fill_between(fpr, tpr, alpha=0.1, color='#2563eb')
    ax.set_xlabel('False Positive Rate', fontsize=13)
    ax.set_ylabel('True Positive Rate', fontsize=13)
    ax.set_title('ROC Curve', fontsize=15, fontweight='bold')
    ax.legend(loc='lower right', fontsize=11)
    ax.grid(True, alpha=0.3)
    fig.savefig(OUTPUTS_DIR / 'roc_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Model does not support probability estimates.')

### 12.3 Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    importances = np.abs(best_model.coef_[0])
else:
    importances = None

if importances is not None:
    feat_imp = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': importances
    }).sort_values('Importance', ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(feat_imp['Feature'], feat_imp['Importance'], color='#2563eb', edgecolor='white')
    ax.set_xlabel('Importance', fontsize=13)
    ax.set_title('Feature Importance', fontsize=15, fontweight='bold')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    fig.savefig(OUTPUTS_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('\nTop 10 Features:')
    print(feat_imp.tail(10)[['Feature', 'Importance']].to_string(index=False))
else:
    print('Feature importance not available for this model type.')

---

## 13. Save Model

In [ ]:
model_path = MODELS_DIR / 'titanic_model.pkl'
joblib.dump(best_model, model_path)
print(f'Model saved to: {model_path}')

# Verify it loads correctly
loaded_model = joblib.load(model_path)
verify_pred = loaded_model.predict(X_test_scaled)
print(f'Verification accuracy: {accuracy_score(y_test, verify_pred):.4f}')

---

## 14. Predictions

In [ ]:
# Generate predictions CSV
y_pred_final = best_model.predict(X_test_scaled)
y_proba_final = best_model.predict_proba(X_test_scaled)[:, 1] if hasattr(best_model, 'predict_proba') else np.full(len(y_pred_final), np.nan)

predictions_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred_final,
    'Probability_Survived': np.round(y_proba_final * 100, 2),
    'Correct': (y_test.values == y_pred_final).astype(int),
})

predictions_df.to_csv(OUTPUTS_DIR / 'predictions.csv', index=False)
print(f'Predictions saved to: {OUTPUTS_DIR / "predictions.csv"}')
print(f'\nCorrect: {predictions_df["Correct"].sum()}/{len(predictions_df)} ({predictions_df["Correct"].mean():.2%})')
predictions_df.head(10)

### Sample Predictions

Let's test the model with some example passengers.

In [ ]:
feature_names = list(X_train.columns)

def predict_passenger(features_dict):
    """Make a prediction for a single passenger."""
    input_data = {feat: features_dict.get(feat, 0) for feat in feature_names}
    X_input = pd.DataFrame([input_data])
    X_input_scaled = pd.DataFrame(scaler.transform(X_input), columns=feature_names)
    
    pred = best_model.predict(X_input_scaled)[0]
    proba = best_model.predict_proba(X_input_scaled)[0] if hasattr(best_model, 'predict_proba') else None
    
    result = 'Survived' if pred == 1 else 'Not Survived'
    prob_str = f'{proba[1]*100:.2f}%' if proba is not None else 'N/A'
    return result, prob_str

# Example 1: 1st class woman
result, prob = predict_passenger({
    'Pclass': 1, 'Sex': 0, 'Age': 29, 'SibSp': 0, 'Parch': 0,
    'Fare': 211.3, 'FamilySize': 1, 'IsAlone': 1, 'CabinAvailable': 1,
    'TicketFrequency': 1, 'Title': 2, 'Embarked_S': 1,
    'FareCategory_VeryHigh': 1, 'AgeCategory_Adult': 1
})
print(f'Passenger 1 (1st Class Female, 29yo): {result} | Probability: {prob}')

# Example 2: 3rd class man
result, prob = predict_passenger({
    'Pclass': 3, 'Sex': 1, 'Age': 22, 'SibSp': 1, 'Parch': 0,
    'Fare': 7.25, 'FamilySize': 2, 'IsAlone': 0, 'CabinAvailable': 0,
    'TicketFrequency': 1, 'Title': 0, 'Embarked_S': 1,
    'FareCategory_Low': 1, 'AgeCategory_Adult': 1
})
print(f'Passenger 2 (3rd Class Male, 22yo):   {result} | Probability: {prob}')

# Example 3: Child
result, prob = predict_passenger({
    'Pclass': 2, 'Sex': 1, 'Age': 5, 'SibSp': 1, 'Parch': 2,
    'Fare': 21.0, 'FamilySize': 4, 'IsAlone': 0, 'CabinAvailable': 0,
    'TicketFrequency': 3, 'Title': 3, 'Embarked_S': 1,
    'FareCategory_Medium': 1, 'AgeCategory_Child': 1
})
print(f'Passenger 3 (2nd Class Boy, 5yo):     {result} | Probability: {prob}')

---

## 15. Conclusion

### Summary

In this project, we built a complete Machine Learning pipeline for the Titanic Survival Prediction task:

1. **Data Exploration:** Thoroughly inspected the dataset (891 passengers, 12 features) and identified missing values in Age (19.9%), Cabin (77.1%), and Embarked (0.2%).

2. **Data Cleaning:** Applied targeted imputation strategies — grouped median for Age, mode for Embarked, and binary flag for Cabin.

3. **Feature Engineering:** Created 7 new features (Title, FamilySize, IsAlone, CabinAvailable, TicketFrequency, FareCategory, AgeCategory) that capture meaningful survival signals.

4. **Model Training:** Trained and compared 6 classification models (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, KNN, SVM).

5. **Hyperparameter Tuning:** Optimized the top 2 models using GridSearchCV with 5-fold cross-validation.

6. **Evaluation:** Achieved strong performance across multiple metrics (Accuracy, Precision, Recall, F1, ROC-AUC).

### Key Findings

- **Gender** was the strongest predictor — females survived at much higher rates.
- **Passenger Class** significantly affected survival — 1st class had the highest survival rate.
- **Age** mattered — children had priority and better survival chances.
- **Fare** (proxy for wealth) correlated positively with survival.
- **Family Size** showed a sweet spot — small families survived better than solo travelers or large families.

### Future Improvements

1. **Advanced Ensembles:** XGBoost, LightGBM, CatBoost for potentially better performance
2. **Stacking/Blending:** Meta-learner ensembles combining multiple model predictions
3. **SHAP Analysis:** Model-agnostic explainability for individual predictions
4. **More Feature Engineering:** Deck extraction from Cabin, surname-based family grouping
5. **Neural Networks:** Simple feedforward network for comparison
6. **Feature Selection:** RFE, Boruta for optimal feature subset
7. **Cross-Validation:** Repeated stratified K-Fold for more robust estimates
8. **Class Imbalance:** SMOTE or class weights if imbalance becomes an issue

---

*Project completed as part of the CodSoft Data Science Internship — Task 1*